# optimizer-loop-on-tensor — worked example 2: Hand-rolled SGD on a tensor with inference_mode

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `optimizer-loop-on-tensor`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A hand-rolled optimizer's `.step()` method must be decorated with `@t.inference_mode()` (or `@t.no_grad()`) to prevent the in-place parameter update from being recorded by autograd. Without this, the update itself would appear in the computation graph, which is both wasteful and incorrect. Inside inference mode, in-place operations on tensors work correctly.

## Worked solution

**Step 1 — Build the hand-rolled optimizer.**
We build `ManualSGD` with `self.params = list(params)` and `self.lr = lr`. The `.step()` method is decorated with `@t.inference_mode()`.

**Step 2 — Perform the update inside inference mode.**
Inside `.step()`, we iterate `self.params`. For each `p` with a non-None `.grad`, we do `p -= self.lr * p.grad`. The `-=` is an in-place operation. Inference mode ensures this doesn't build a Recipe/autograd node.

**Step 3 — Compare to PyTorch's SGD.**
We run both optimizers from the same initial tensor for 8 steps on the same sequence of losses. The trajectories must be identical.

**Step 4 — Verify no autograd side-effects.**
After the `.step()` call, the tensor should still have `requires_grad=True` — the parameter itself is still trainable; only the update step was excluded from the graph.

In [ ]:
import torch as t

class ManualSGD:
    def __init__(self, params, lr):
        self.params = list(params)
        self.lr = lr

    @t.inference_mode()
    def step(self):
        for p in self.params:
            if p.grad is not None:
                p -= self.lr * p.grad

    def zero_grad(self):
        for p in self.params:
            p.grad = None

# --- compare against PyTorch SGD ---
t.manual_seed(9)
xy_a = t.tensor([2.0, -3.0], requires_grad=True)
xy_b = xy_a.clone().detach().requires_grad_(True)

opt_manual = ManualSGD([xy_a], lr=0.05)
opt_torch  = t.optim.SGD([xy_b], lr=0.05)

for _ in range(8):
    opt_manual.zero_grad()
    ((xy_a ** 2).sum()).backward()
    opt_manual.step()

    opt_torch.zero_grad()
    ((xy_b ** 2).sum()).backward()
    opt_torch.step()

print(f'ManualSGD: {xy_a.data.tolist()}')
print(f'PyTorch  : {xy_b.data.tolist()}')
assert t.allclose(xy_a.data, xy_b.data, atol=1e-5), 'Trajectories should be identical'
print('Trajectories match!')
assert xy_a.requires_grad, 'Tensor should still require grad after step'